# Reasoning Traces Analysis - Multi-Model Comparison

Compare reasoning trace lengths and prediction results across multiple models

## 1. Import Required Libraries

In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from transformers import AutoTokenizer

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (14, 6)

## 2. Load Model Results Data

In [ ]:
# Load results from all available models
base_dir = Path("data/reasoning_traces")
model_dirs = sorted([d for d in base_dir.iterdir() if d.is_dir()])

print(f"Found {len(model_dirs)} models:")
for d in model_dirs:
    print(f"  - {d.name}")

# Load data for each model
models_data = {}
for model_dir in model_dirs:
    model_name = model_dir.name
    parsed_file = model_dir / "parsed_results.json"
    
    if parsed_file.exists():
        with open(parsed_file, "r") as f:
            results = json.load(f)
        models_data[model_name] = pd.DataFrame(results)
        print(f"\n{model_name}: {len(results)} samples")
        if len(results) > 0:
            print(f"  Columns: {list(results[0].keys())}")

print(f"\nTotal models with data: {len(models_data)}")

## 3. Calculate Token Counts

In [ ]:
# Load tokenizer and compute token counts for reasoning
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2-7B")

for model_name, df in models_data.items():
    df["reasoning_tokens"] = df["reasoning"].apply(
        lambda x: len(tokenizer.encode(x)) if isinstance(x, str) else 0
    )
    print(f"\n{model_name} - Reasoning Token Stats:")
    print(f"  Mean: {df['reasoning_tokens'].mean():.0f}")
    print(f"  Median: {df['reasoning_tokens'].median():.0f}")
    print(f"  Min-Max: {df['reasoning_tokens'].min()} - {df['reasoning_tokens'].max()}")

## 4. Compare Prediction Performance

In [ ]:
# Calculate accuracy for each model and condition
performance_metrics = []

for model_name, df in models_data.items():
    for condition in df["condition"].unique():
        subset = df[df["condition"] == condition]
        
        # Check if we have ground truth labels
        if "ground_truth" in subset.columns and "predicted" in subset.columns:
            # Calculate accuracy across three fields
            all_correct = (
                (subset["predicted_prompt_harm"] == subset.get("ground_truth_prompt_harm", None)) &
                (subset["predicted_response_harm"] == subset.get("ground_truth_response_harm", None)) &
                (subset["predicted_response_refusal"] == subset.get("ground_truth_response_refusal", None))
            ).sum() / len(subset)
            
            performance_metrics.append({
                "model": model_name,
                "condition": condition,
                "samples": len(subset),
                "avg_tokens": subset["reasoning_tokens"].mean(),
                "accuracy": all_correct
            })

perf_df = pd.DataFrame(performance_metrics)
if len(perf_df) > 0:
    print("\nPrediction Performance by Model and Condition:")
    print(perf_df.to_string(index=False))
else:
    print("\nNo prediction data available yet")

## 5. Visualize Results

In [ ]:
# Create comparison visualizations
if len(models_data) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    
    # Plot 1: Average tokens by model
    token_data = [(name, df["reasoning_tokens"].mean()) for name, df in models_data.items()]
    names, tokens = zip(*token_data)
    axes[0].bar(range(len(names)), tokens, color="steelblue")
    axes[0].set_xticks(range(len(names)))
    axes[0].set_xticklabels(names, rotation=45, ha="right")
    axes[0].set_ylabel("Average Reasoning Token Count")
    axes[0].set_title("Reasoning Length Comparison")
    axes[0].grid(axis="y", alpha=0.3)
    
    # Plot 2: Token count distribution
    for model_name, df in models_data.items():
        axes[1].hist(df["reasoning_tokens"], label=model_name, alpha=0.6, bins=20)
    axes[1].set_xlabel("Reasoning Token Count")
    axes[1].set_ylabel("Frequency")
    axes[1].set_title("Token Count Distribution")
    axes[1].legend()
    
    plt.tight_layout()
    plt.show()
else:
    print("No data available to visualize")

## 6. Summary Statistics

In [ ]:
# Generate summary statistics
summary_stats = []

for model_name, df in models_data.items():
    summary_stats.append({
        "Model": model_name,
        "Total Samples": len(df),
        "Avg Tokens": f"{df['reasoning_tokens'].mean():.0f}",
        "Median Tokens": f"{df['reasoning_tokens'].median():.0f}",
        "Min Tokens": df["reasoning_tokens"].min(),
        "Max Tokens": df["reasoning_tokens"].max(),
        "Std Dev": f"{df['reasoning_tokens'].std():.0f}"
    })

summary_df = pd.DataFrame(summary_stats)
print("\nSummary Statistics:")
print(summary_df.to_string(index=False))

In [ ]:
# Load tokenizer and compute token counts for reasoning
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2-7B")

for model_name, df in models_data.items():
    df["reasoning_tokens"] = df["reasoning"].apply(
        lambda x: len(tokenizer.encode(x)) if isinstance(x, str) else 0
    )
    print(f"\n{model_name} - Reasoning Token Stats:")
    print(f"  Mean: {df['reasoning_tokens'].mean():.0f}")
    print(f"  Median: {df['reasoning_tokens'].median():.0f}")
    print(f"  Min-Max: {df['reasoning_tokens'].min()} - {df['reasoning_tokens'].max()}")

In [ ]:
# Load results from all available models
base_dir = Path("data/reasoning_traces")
model_dirs = sorted([d for d in base_dir.iterdir() if d.is_dir()])

print(f"Found {len(model_dirs)} models:")
for d in model_dirs:
    print(f"  - {d.name}")

# Load data for each model
models_data = {}
for model_dir in model_dirs:
    model_name = model_dir.name
    parsed_file = model_dir / "parsed_results.json"
    
    if parsed_file.exists():
        with open(parsed_file, "r") as f:
            results = json.load(f)
        models_data[model_name] = pd.DataFrame(results)
        print(f"\n{model_name}: {len(results)} samples")
        if len(results) > 0:
            print(f"  Conditions: {results[0]['condition']}")

print(f"\nTotal models with data: {len(models_data)}")

## 3. Calculate Token Counts

In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from transformers import AutoTokenizer

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (14, 6)

## 2. Load Model Results Data

# Reasoning Traces Analysis - Multi-Model Comparison

Compare reasoning trace lengths and prediction results across multiple models

## 1. Import Required Libraries

# Reasoning Traces Analysis

View reasoning traces and their token lengths using the Qwen tokenizer.

In [ ]:
import json
import pandas as pd
from pathlib import Path
from transformers import AutoTokenizer

In [ ]:
# Load parsed results
data_dir = Path("data/reasoning_traces")
parsed_results_file = data_dir / "parsed_results.json"

with open(parsed_results_file, "r") as f:
    results = json.load(f)

df = pd.DataFrame(results)
print(f"Loaded {len(df)} samples")
print(f"Conditions: {df['condition'].value_counts().to_dict()}")

In [ ]:
# Load Qwen tokenizer and calculate token counts for reasoning traces
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2-7B")

df["reasoning_tokens"] = df["reasoning"].apply(
    lambda x: len(tokenizer.encode(x)) if isinstance(x, str) else 0
)

for condition in df["condition"].unique():
    subset = df[df["condition"] == condition]
    print(f"\n{condition} - Reasoning token count stats:")
    print(f"  Mean: {subset['reasoning_tokens'].mean():.0f}")
    print(f"  Min:  {subset['reasoning_tokens'].min()}")
    print(f"  Max:  {subset['reasoning_tokens'].max()}")
    print(f"  Median: {subset['reasoning_tokens'].median():.0f}")

In [ ]:
# Display responses grouped by sample, showing both conditions side by side
without_df = df[df["condition"] == "without_intent"].reset_index(drop=True)
with_df = df[df["condition"] == "with_intent"].reset_index(drop=True)

for idx in range(len(without_df)):
    wo = without_df.iloc[idx]
    wi = with_df.iloc[idx]
    
    print(f"\n{'='*80}")
    print(f"Sample {idx + 1} (Wildguard ID: {wo['wildguard_id']})")
    print(f"{'='*80}")
    
    print(f"\nPrompt:\n  {wo['prompt'][:200]}{'...' if len(wo['prompt']) > 200 else ''}")
    print(f"\nIntent:\n  {wo['intent']}")
    print(f"\nResponse:\n  {wo['response'][:200]}{'...' if len(wo['response']) > 200 else ''}")
    
    print(f"\n--- Without Intent ({wo['reasoning_tokens']} tokens) ---")
    print(f"  Predicted: {wo['predicted']}")
    
    print(f"\n--- With Intent ({wi['reasoning_tokens']} tokens) ---")
    print(f"  Predicted: {wi['predicted']}")
    
    print(f"\n  Ground truth: {wo['ground_truth']}")